# Training Curves — NYUv2 Depth LoRA (50 epochs)
Parses `log.txt` in real-time and plots loss + learning rate curves.
Re-run any cell to refresh while training is still running.

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

LOG_PATH = '../output/depth/nyu_rgb_r8/log.txt'

In [2]:
# --- Parse log.txt (JSON Lines: one dict per epoch) ---
rows = []
with open(LOG_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            pass

df = pd.DataFrame(rows)
df = df.rename(columns={
    'train/lr':                 'lr',
    'train/loss':               'train_loss',
    'train/loss_tok_depth@224': 'depth_loss',
    'train/grad_norm':          'grad_norm',
    'val/loss':                 'val_loss',
    'val/token_acc':            'val_acc',
})

print(f"Epochs in log : {len(df)}")
if not df.empty:
    last = df.iloc[-1]
    print(f"Last epoch    : {int(last.epoch)}")
    print(f"Train loss    : {last.train_loss:.4f}")
    print(f"LR            : {last.lr:.2e}")
    if 'val_loss' in df.columns and pd.notna(last.get('val_loss')):
        print(f"Val loss      : {last.val_loss:.4f}")
df.head()

Epochs in log : 50
Last epoch    : 49
Train loss    : 2.2788
LR            : 1.02e-08
Val loss      : 13.9173


,lr,train_loss,train/loss_tok_rgb@224,depth_loss,grad_norm,val_loss,val_acc,epoch,lora_rank,lora_modules
0,0.000002,3.630208,0.0,7.260416,3.407305,9.361869,0.00224,0,8,attention
1,0.000007,3.137909,0.0,6.275817,3.618522,NaN,NaN,1,8,attention
2,0.000013,2.990375,0.0,5.980749,5.031438,NaN,NaN,2,8,attention
3,0.000018,2.900579,0.0,5.801157,6.240626,NaN,NaN,3,8,attention
4,0.000023,2.830645,0.0,5.661290,6.021561,NaN,NaN,4,8,attention


In [3]:
# --- Plot: Loss curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NYUv2 Depth — LoRA fine-tuning (rank=8, XL)', fontsize=13)

ax = axes[0]
ax.plot(df['epoch'], df['train_loss'], 'o-', color='steelblue', lw=2, ms=4, label='train loss')
val_df = df.dropna(subset=['val_loss'])
if not val_df.empty:
    ax.plot(val_df['epoch'], val_df['val_loss'], 's--', color='tomato', lw=2, ms=6, label='val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Train vs Val loss (all tokens)')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend()
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(df['epoch'], df['depth_loss'], 'o-', color='darkorchid', lw=2, ms=4, label='train depth loss')
if not val_df.empty:
    ax2.plot(val_df['epoch'], val_df['val_loss'] * 2, 's--', color='tomato', lw=2, ms=6, label='val depth loss (≈ val_loss × 2)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Cross-entropy loss (depth tokens)')
ax2.set_title('Depth token loss')
ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/depth/nyu_rgb_r8/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → output/depth/nyu_rgb_r8/loss_curves.png')

Saved → output/depth/nyu_rgb_r8/loss_curves.png


In [4]:
# --- Plot: Learning rate schedule ---
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df['epoch'], df['lr'], 'o-', color='darkorange', lw=2, ms=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning rate')
ax.set_title('Learning rate schedule (cosine + warmup)')
ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/depth/nyu_rgb_r8/lr_schedule.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → output/depth/nyu_rgb_r8/lr_schedule.png')

Saved → output/depth/nyu_rgb_r8/lr_schedule.png
